### Activation Environment and Import Packages

In [1]:
using Pkg
Pkg.activate("./env")
using Plots
using DataFrames
using ImportAll
using CSV
@importall(MarinePowerDynamics)

  Activating project at `~/research/MarinePowerDynamics.jl/example/env`


In [6]:
function plot_gens(sol)
    generator_indices = findall(bus -> typeof(bus) == FourthOrderEq || typeof(bus) == LinearPTO, powergrid.nodes)
    labels = reshape(generator_indices,(1,length(generator_indices)))

    pl_v = plot(sol, generator_indices, :v, legend = (0.8, 0.7), ylabel="V [p.u.]",label = labels)
    pl_p = plot(sol, generator_indices, :p, legend = (0.8, 0.7), ylabel="p [p.u.]", label=labels)
    #pl_q = plot(sol, generator_indices, :q, legend = (0.8, 0.7), ylabel="q [p.u.]", label=labels)
    pl_ω = plot(sol, generator_indices, :ω, legend = (0.8, 0.7), ylabel="ω [rad/s]", label=labels)
    pl_φ = plot(sol, generator_indices, :φ, legend = (0.8, 0.7), ylabel="φ [rad]", label=labels)

    pl = plot(pl_ω, pl_v, pl_p, pl_φ;
            layout=(2,2),
            size = (1000, 500),
            lw=3,
            xlabel="t[s]")

end

function save_generator_data_comprehensive(solution, powergrid; output_dir="./simulation_results")
    if !isdir(output_dir)
        mkpath(output_dir)  # Use mkpath to create nested directories
    end
    
    generator_indices = findall(bus -> typeof(bus) == FourthOrderEq || typeof(bus) == LinearPTO, powergrid.nodes)
    time_vector = solution.dqsol.t
    
    for label in generator_indices
        # Create DataFrame starting with time
        df = DataFrame(time = time_vector)
        
        # Add each variable as a column
        for var in [:v, :ω, :p, :φ]
            data = [solution(t, label, var) for t in time_vector]
            df[!, String(var)] = data
        end
        
        # Save comprehensive data for this generator
        filename = "$(label)_all_variables.csv"
        CSV.write(joinpath(output_dir, filename), df)
        
        println("Saved comprehensive data for $label to $filename")
    end
end


save_generator_data_comprehensive (generic function with 2 methods)

In [2]:
build_grids()

## Flat Run

In [7]:
### flat run test
powergrid = read_powergrid(joinpath(@__DIR__,"./src/IEEE14grid.json"), Json)
operationpoint = find_operationpoint(powergrid)

# Configuration for the simulation
# notice the fault span is after the simulation start time to run a flat run
config = Dict(
    "fault_span" => (61, 62),
    "component" => "branch1",
    "timespan" => (0.0, 60.0),
    "description" => "IEEE14 flat run - no fault during simulation time",
    "grid_type" => "IEEE14_standard"
)
fault = LineFailure(line_name = config["component"], tspan_fault = config["fault_span"])

# run dynamic simulation
solution = simulate(fault, powergrid, operationpoint, config["timespan"])

#plot_gens(solution)
save_generator_data_comprehensive(solution, powergrid, output_dir="./simulation_results/flat/no_wec")

Saved comprehensive data for bus1 to bus1_all_variables.csv
Saved comprehensive data for bus3 to bus3_all_variables.csv
Saved comprehensive data for bus6 to bus6_all_variables.csv
Saved comprehensive data for bus8 to bus8_all_variables.csv
Saved comprehensive data for bus3 to bus3_all_variables.csv
Saved comprehensive data for bus6 to bus6_all_variables.csv
Saved comprehensive data for bus8 to bus8_all_variables.csv


In [ ]:
### flat run test
powergrid = read_powergrid(joinpath(@__DIR__,"./src/WEC_IEEE14grid.json"), Json)
operationpoint = find_operationpoint(powergrid)

# Configuration for the simulation
# notice the fault span is after the simulation start time to run a flat run
config = Dict(
    "fault_span" => (61, 62),
    "component" => "branch1",
    "timespan" => (0.0, 60.0),
    "description" => "IEEE14 flat run - no fault during simulation time",
    "grid_type" => "IEEE14_wec"
)
fault = LineFailure(line_name = config["component"], tspan_fault = config["fault_span"])

# run dynamic simulation
solution = simulate(fault, powergrid, operationpoint, config["timespan"])

#plot_gens(solution)
save_generator_data_comprehensive(solution, powergrid, output_dir="./simulation_results/flat/wec")

## Faults 

In [ ]:
### Step Change - Regular IEEE 14 
powergrid = read_powergrid(joinpath(@__DIR__,"./src/IEEE14grid.json"), Json)
operationpoint = find_operationpoint(powergrid)

# Configuration for the simulation
config = Dict(
    "fault_span" => (30, 45),
    "component" => "bus8",
    "timespan" => (0.0, 60.0),
    "description" => "IEEE14 step change at bus 8 at 30s and cleared at 45s. increase from 0 to 1 pu",
    "grid_type" => "IEEE14_standard"
)

fault = NodeParameterChange(node=config["component"], value=1.0, tspan_fault=config["fault_span"], var=:P)


# run dynamic simulation
solution = simulate(fault, powergrid, operationpoint, config["timespan"])

#plot_gens(solution)
save_generator_data_comprehensive(solution, powergrid, output_dir="./simulation_results/fault/no_wec")

In [ ]:
### Step Change - WEC IEEE 14 
powergrid = read_powergrid(joinpath(@__DIR__,"./src/WEC_IEEE14grid.json"), Json)
operationpoint = find_operationpoint(powergrid)

# Configuration for the simulation
config = Dict(
    "fault_span" => (30, 45),
    "component" => "bus8",
    "timespan" => (0.0, 60.0),
    "description" => "IEEE14 step change at bus 8 at 30s and cleared at 45s. increase from 0 to 1 pu",
    "grid_type" => "IEEE14_wec"
)

fault = PowerPerturbation(node=config["component"], fault_power=1.0, tspan_fault=config["fault_span"], var=:P)


# run dynamic simulation
solution = simulate(fault, powergrid, operationpoint, config["timespan"])
#plot_gens(solution)
save_generator_data_comprehensive(solution, powergrid, output_dir="./simulation_results/fault/wec")